# Rung 10 — Self-consistency: majority vote over k sampled `number` answers

> ⚠️ **NOT RUN YET.** Cells below are the pod plan. Run them in order and **stop where marked.**

## Pre-registered rule — WRITTEN BEFORE ANY RESULT

Judged on `number` only, disaggregated ID/OOD, read as **margin** over the template-aware floor.

| Outcome | Verdict |
|---|---|
| T0 `mode_share ≈ 1.0` (no diversity) | **DEAD** — faithful negative, run nothing further |
| T0 diverse but the mode sits on the same biased value | **DEAD** — voting reinforces the error |
| `number` rises **≥ +0.02**, CI excludes 0, in ID **and** OOD | **PASS** → candidate |
| rises in one distribution only | **PARTIAL** — not promoted |
| flat or down | **FAITHFUL NEGATIVE** |

🔴 **The threshold does not move afterwards.** `bucket_mean` does NOT decide this rung — it
averages four buckets while this changes one format inside one of them.

🔴 **Why T0 comes first.** Voting moves toward the mode, and ours is *measured biased*: 05b found
81.5% of `number` errors are under-counts; 05c found true 2, 3 and 4 share the same modal
prediction (1). Aggregating a biased distribution reinforces it — the mechanism that killed
calibration. The decisive column is **`mode_closer_than_greedy`**, not entropy.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parents[1]
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "experiments" / "10-self-consistency"))

from _models.vote import diversity, t0_diversity, vote_number
from frame.config import BaselineConfig
from frame.parsing import parse_number

RUN_DIR = REPO / "experiments/10-self-consistency/runs/10_self_consistency_v1"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CONTROL = REPO / "experiments/06-vit-lora/runs/06_vit_lora_v1/eval_best/inspect.csv"

## 1. 🔴 GATE — flag-OFF must be byte-identical

There is no local GPU, so this could not be checked before the pod. **If a single string differs,
STOP:** the flag is not defaulting OFF and the A/B is invalid.

In [ ]:
# load the rung 06 checkpoint with n_samples=1 and re-predict 50 questions;
# every string must equal the corresponding entry in rung 06 predictions.json.
# STOP HERE ON ANY MISMATCH.

## 2. T0 — is there anything to vote on? (~20 min)

300 `number` questions stratified by template; `temperature ∈ {0.3, 0.7, 1.0}`, k=8.
**Report and STOP.** Do not continue to T1 without an explicit go.

In [ ]:
# sample with cfg.n_samples=8 at each temperature; keep the RAW samples
# t0 = t0_diversity(records); t0.to_csv(RUN_DIR / "t0_diversity.csv", index=False)
# t0

### 🛑 STOP — report T0 and wait for a go/no-go before T1

## 3. T1 — the voting run (only with a go). k and temperature are FIXED BY T0.

In [ ]:
# all 2094 `number` questions at the k/temperature T0 selected
# save samples.json RAW — rung 05 discarded its predicted text and 05b had to re-derive it

## 4. T2 — scoring (zero GPU)

The control is **not** re-run — but it **is** validated first.

In [ ]:
ctrl = pd.read_csv(CONTROL)
ctrl = ctrl[ctrl.answer_format == "number"]
acc_ctrl = float(ctrl.correct.mean())
assert len(ctrl) == 2094, f"expected 2094 number rows, got {len(ctrl)}"
assert abs(acc_ctrl - 0.4250) < 0.005, f"control does not reproduce rung 06: {acc_ctrl:.4f}"
print(f"control validated: acc={acc_ctrl:.4f} over n={len(ctrl)}")

In [ ]:
# score the voted arm with frame.metrics — never re-derive metrics here (RULES §EVAL)
# disaggregate ID/OOD; report MARGIN over the template-aware floor, not raw accuracy
# write RESULTS.csv: k, temperature, acc_number_ID, acc_number_OOD, margins, CI, n